Dataset

In [59]:
import kagglehub
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os
# Download latest version
path = kagglehub.dataset_download("dansbecker/cityscapes-image-pairs")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'cityscapes-image-pairs' dataset.
Path to dataset files: /kaggle/input/cityscapes-image-pairs


In [60]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split, ConcatDataset
import os
class CityScapes(Dataset):
    def __init__(self, root, joint_transforms=None, pic_transform=None, split="train"):
        # super().__init__()
        self.path = os.path.join(root, "train")
        self.dataset = os.listdir(self.path)
        self.pic_transform = pic_transform
        self.joint_transforms = joint_transforms

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image_name = self.dataset[index]
        image = cv2.imread(os.path.join(self.path, image_name))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        H, W, C = image.shape
        W2 = W // 2
        pic = image[:, :W2, :]
        mask = image[:, W2:, :]
        if self.joint_transforms is not None:
            pic = self.joint_transforms(pic)
            mask = self.joint_transforms(mask)
        if self.pic_transform is not None:
            pic = self.pic_transform(pic)
        return pic, torch.tensor(mask)

In [61]:
from torchvision import transforms
test_joint_transform = None
test_transform = transforms.Compose([
        transforms.ToTensor(),


])

new_path = os.path.join(path, "cityscapes_data")
train_val_dataset = CityScapes(new_path, test_joint_transform, test_transform, "train")
test_dataset = CityScapes(new_path, test_joint_transform, test_transform, "val")
train_size = int(len(train_val_dataset)*0.75)
val_size = len(train_val_dataset) - train_size
train_dataset, val_dataset = random_split(train_val_dataset, [train_size, val_size])
f"train : {len(train_dataset)} | val: {len(val_dataset)} | test : {len(test_dataset)}"

'train : 2231 | val: 744 | test : 2975'

Augmentation

In [62]:
class Augment_Img(Dataset):
    def __init__(self, dataset, pic_transforms=None, joint_transforms=None):
        self.dataset = dataset
        self.pic_transforms = pic_transforms
        self.joint_transforms = joint_transforms

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image, mask = self.dataset[index]
        if self.pic_transforms is not None:
            image = self.pic_transforms(image)
        if self.joint_transforms is not None:
            mask = self.joint_transforms(mask)
            image = self.joint_transforms(image)
        return image, mask

train_trans1 = transforms.Compose([
    transforms.ColorJitter(0, 0, 0.1),
])

train_trans2 = transforms.Compose([
    transforms.GaussianBlur((3, 3)),
])
train_trans3 = transforms.Compose([
    transforms.Grayscale(3)
])
joint_trans3 = transforms.Compose([
    transforms.CenterCrop((200, 200)),
    transforms.Resize((256, 256))
])
train_dataset1 = Augment_Img(train_dataset, train_trans1)
train_dataset2 = Augment_Img(train_dataset, train_trans2)
train_dataset3 = Augment_Img(train_dataset, train_trans3, joint_trans3)
aug_train_dataset = ConcatDataset([train_dataset, train_dataset1, train_dataset2, train_dataset3])
f"train : {len(aug_train_dataset)} | val: {len(val_dataset)} | test : {len(test_dataset)}"

'train : 8924 | val: 744 | test : 2975'

Config

In [63]:
from torch import optim
class Config:
    def __init__(self):
        self.batch_size = 32
        self.lr = 1e-3
        self.weight_decay = 1e-4
        self.n_epochs = 5


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = Config()

DataLoader

In [70]:

train_loader = DataLoader(aug_train_dataset, config.batch_size, True, pin_memory=True)
val_loader = DataLoader(val_dataset, config.batch_size, False, pin_memory=True)
train_loader = DataLoader(test_dataset, config.batch_size, False, pin_memory=True)

In [65]:
from torch import nn
from torchvision.models import VGG11_BN_Weights, vgg11_bn
class SAM(nn.Module):
    def __init__(self):
        super().__init__()
        model = vgg11_bn(weights=VGG11_BN_Weights.DEFAULT)
        self.backbone = model.features

        self.encoder1 = self.backbone[0:4] # 256 -> 128
        self.encoder2 = self.backbone[4:8] # 128 -> 64
        self.encoder3 = self.backbone[8:15] # 64 -> 32
        self.encoder4 = self.backbone[15:22] # 32 -> 16

        self.bottelneck = nn.Conv2d(512, 512, 1)

        self.upsample = nn.Upsample(scale_factor=2, align_corners=True, mode="bilinear")
        self.decoder1 = self._conv_block(256 + 512, 256) # 16 -> 32
        self.decoder2 = self._conv_block(128 + 256, 128) # 32 -> 64
        self.decoder3 = self._conv_block(64 + 128, 64) # 64 -> 128
        self.decoder4 = self._conv_block(64, 32) # 128 -> 256
        self.out_layer = nn.Conv2d(32, 3, 1)


    def _conv_block(self, input_size, output):
        return nn.Sequential(
            nn.Conv2d(input_size, output, 3, padding=1),
            nn.BatchNorm2d(output),
            nn.PReLU(),
            nn.Dropout2d(0.1),
        )
    def forward(self, x):

        e1 = self.encoder1(x) # 256 -> 128 64
        e2 = self.encoder2(e1) # 128 -> 64 128
        e3 = self.encoder3(e2) # 64 -> 32 256
        e4 = self.encoder4(e3) # 32 -> 16 512

        b = self.bottelneck(e4)

        d1 = self.decoder1(torch.cat([self.upsample(b), e3], dim=1))    # 16 -> 32
        d2 = self.decoder2(torch.cat([self.upsample(d1), e2], dim=1))    # 32 -> 64
        d3 = self.decoder3(torch.cat([self.upsample(d2), e1], dim=1))    # 64 -> 128
        d4 = self.decoder4(self.upsample(d3))    # 128 -> 256

        return self.out_layer(d4)


Freeze some layers

In [66]:
model_sam = SAM()
model_sam.encoder1.requires_grad_(False)
model_sam.encoder2.requires_grad_(False)
model_sam.encoder3.requires_grad_(False)
model_sam.encoder1.eval()
model_sam.encoder2.eval()
model_sam.encoder3.eval();

In [67]:
def confusion_matrix(num_classes, pred, true):
    cm = torch.zeros(num_classes, num_classes)

    for i, j in zip(true, pred):
        cm[i, j] += 1
    tp = torch.diag(cm)
    pred_pos = cm.sum(dim=0)
    actual_pos = cm.sum(dim=1)

    precision = (tp / pred_pos).mean()
    recalln = (tp / actual_pos).mean()
    fn = actual_pos - tp
    miou = tp / (pred_pos + fn)

    acc = tp.sum() / cm.sum()
    return acc , miou




In [73]:
from torch import optim
from tqdm.auto import tqdm, trange
from sklearn.metrics import r2_score
model_sam = model.to(device, non_blocking=True)
optimizer = optim.AdamW(model_sam.parameters(), config.lr, weight_decay=config.weight_decay)
# model = torch.compile(model.to(device, non_blocking=True))
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.n_epochs)
loss_func = nn.MSELoss()
grad_scaler = torch.amp.GradScaler()
epoch_bar = trange(config.n_epochs)
train_bar = tqdm(aug_train_dataset)
val_bar = tqdm(val_dataset)
history_classification = {"miou_train": [],
           "miou_val" : [],
           "val_loss" : [],
           "train_loss": [],
           "val_acc" : [],
           "train_acc" : []}
history_regresion = {"r2_train": [],
           "r2_val" : [],
           "val_loss" : [],
           "history_regresion": []}
for epoch in epoch_bar:
    runnin_train_loss = 0.0
    running_val_loss = 0.0
    for x_train, y_train in train_bar:
        print(x_train.shape)
        optimizer.zero_grad()
        x_train = x_train.to(device, non_blocking=True)
        y_train = y_train.to(device, non_blocking=True)
        with torch.amp.autocast(device.type):
            train_pred = model_sam(x_train)
            train_loss = loss_func(train_pred, y_train)
        grad_scaler.scale(train_loss).backward()
        nn.utils.clip_grad_norm_(model_sam.parameters(), 2)
        grad_scaler.unscale_(optimizer)
        grad_scaler.step(optimizer)
        grad_scaler.update()
        runnin_train_loss += train_loss.item()
        train_var.set_postfix(loss=f"{train_loss.item():.4f}")
        history_regresion["train_loss"].append(train_loss.item())
        history_regresion["r2_train"].append(train_pred.detach().cpu().numpy(), y_train.detach().cpu().numpy())

    scheduler.step()
    with torch.inference_mode():
        for x_val, y_val in val_bar:
            with torch.amp.autocast(device.type):
                val_pred = model(x_val)
                val_loss = loss_func(val_pred, y_val)
            runnin_val_loss += val_loss.item()
            val_bar.set_postfix(loss=f"{val_loss.item():.4f}")
            history_regresion["val_loss"].append(val_loss.item())
            history_regresion["r2_val"].append(val_pred.detach().cpu().numpy(), y_val.detach().cpu().numpy())

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/8924 [00:00<?, ?it/s]

  0%|          | 0/744 [00:00<?, ?it/s]

torch.Size([3, 256, 256])


ValueError: expected 4D input (got 3D input)